# agentic-civic-resolution-app
# Day 1 | Notebook 3: Synthetic Complaint Generator
*Goal:** Generate realistic synthetic 311 complaints that mirror NYC 311 schema.

Use cases:
Dev/test when API rate limits are hit
Load testing (generate 100k+ rows instantly)
Simulate edge cases: missing geo, extreme backlog, new complaint types
Privacy-safe demo data

## 0. Config

In [0]:
# Databricks notebook source

CATALOG         = "civicops"
SILVER_TABLE    = f"{CATALOG}.silver.complaints"
SYNTHETIC_TABLE = f"{CATALOG}.silver.complaints_synthetic"


NUM_RECORDS     = 50_000     # change to 500_000 for load testing
SEED            = 42


## 1. Reference Data

In [0]:

import random
import string
from datetime import datetime, timedelta
import pandas as pd
import numpy as np

random.seed(SEED)
np.random.seed(SEED)

# ── Complaint types with realistic weights ─────────────────────────────────
COMPLAINT_TYPES = {
    "NOISE - RESIDENTIAL":         0.18,
    "HEAT/HOT WATER":              0.12,
    "ILLEGAL PARKING":             0.10,
    "BLOCKED DRIVEWAY":            0.08,
    "STREET LIGHT CONDITION":      0.07,
    "PAINT/PLASTER":               0.06,
    "WATER SYSTEM":                0.05,
    "PLUMBING":                    0.05,
    "RODENT":                      0.04,
    "NOISE - COMMERCIAL":          0.04,
    "UNSANITARY CONDITION":        0.04,
    "ELEVATOR":                    0.03,
    "GRAFFITI":                    0.03,
    "OVERGROWN TREE/BRANCHES":     0.03,
    "DAMAGED TREE":                0.03,
    "TAXI COMPLAINT":              0.02,
    "DERELICT VEHICLES":           0.02,
    "AIR QUALITY":                 0.01,
}

BOROUGHS = {
    "BROOKLYN":   0.31,
    "QUEENS":     0.27,
    "MANHATTAN":  0.20,
    "BRONX":      0.17,
    "STATEN ISLAND": 0.05,
}

# Borough → rough lat/lon centers + spread
BOROUGH_GEO = {
    "BROOKLYN":      (40.6501, -73.9496, 0.08),
    "QUEENS":        (40.7282, -73.7949, 0.10),
    "MANHATTAN":     (40.7831, -73.9712, 0.05),
    "BRONX":         (40.8448, -73.8648, 0.07),
    "STATEN ISLAND": (40.5795, -74.1502, 0.06),
}

AGENCIES = {
    "HPD":  ["HEAT/HOT WATER", "PAINT/PLASTER", "PLUMBING", "ELEVATOR", "UNSANITARY CONDITION"],
    "DOT":  ["STREET LIGHT CONDITION", "ILLEGAL PARKING", "BLOCKED DRIVEWAY", "DAMAGED TREE"],
    "NYPD": ["NOISE - RESIDENTIAL", "NOISE - COMMERCIAL", "ILLEGAL PARKING", "TAXI COMPLAINT"],
    "DSNY": ["RODENT", "UNSANITARY CONDITION", "GRAFFITI", "DERELICT VEHICLES"],
    "DEP":  ["WATER SYSTEM", "AIR QUALITY"],
    "DPR":  ["OVERGROWN TREE/BRANCHES", "DAMAGED TREE", "GRAFFITI"],
}

# Build complaint_type → agency lookup
COMPLAINT_AGENCY = {}
for agency, types in AGENCIES.items():
    for t in types:
        COMPLAINT_AGENCY[t] = agency

STATUSES       = ["OPEN", "CLOSED", "PENDING", "IN PROGRESS"]
STATUS_WEIGHTS = [0.25,   0.55,    0.12,       0.08]

DESCRIPTORS = {
    "NOISE - RESIDENTIAL": ["Loud music/party", "Banging/hammering", "Loud television", "People shouting"],
    "HEAT/HOT WATER":      ["Entire building", "Apartment only", "Radiator issue", "No hot water"],
    "ILLEGAL PARKING":     ["Blocking fire hydrant", "Blocking crosswalk", "Double parked", "No standing zone"],
    "BLOCKED DRIVEWAY":    ["Partial access", "No access", "Vehicle across driveway"],
    "STREET LIGHT CONDITION": ["Street light out", "Dim light", "Flickering", "Entire block out"],
    "DEFAULT":             ["Complaint filed", "Inspection needed", "Follow-up required"],
}

RESOLUTION_TEMPLATES = [
    "The condition was inspected and corrected.",
    "Upon inspection, the condition was found to be in compliance.",
    "The condition was not observed at time of inspection.",
    "The agency responsible has been notified.",
    "Repair work has been scheduled.",
    "The issue has been resolved by the property owner.",
    None,  # unresolved / open
]

## 2. Generator Functions

In [0]:

def random_complaint_id(n: int = 8) -> str:
    return "SYN-" + "".join(random.choices(string.digits, k=n))


def random_date(start_days_ago: int = 730, end_days_ago: int = 0) -> datetime:
    delta = random.randint(end_days_ago, start_days_ago)
    return datetime.now() - timedelta(days=delta, hours=random.randint(0, 23),
                                      minutes=random.randint(0, 59))


def generate_record() -> dict:
    complaint_type = random.choices(
        list(COMPLAINT_TYPES.keys()), weights=list(COMPLAINT_TYPES.values())
    )[0]

    borough = random.choices(
        list(BOROUGHS.keys()), weights=list(BOROUGHS.values())
    )[0]

    status = random.choices(STATUSES, weights=STATUS_WEIGHTS)[0]
    created = random_date(start_days_ago=730)

    is_closed = status == "CLOSED"
    closed_date = None
    days_open = None
    if is_closed:
        resolution_days = max(0, int(np.random.exponential(scale=8)))
        closed_date = created + timedelta(days=resolution_days)
        days_open = resolution_days
    else:
        days_open = (datetime.now() - created).days

    lat_c, lon_c, spread = BOROUGH_GEO[borough]
    has_geo = random.random() > 0.05   # 95% have geo
    lat = round(lat_c + np.random.normal(0, spread * 0.4), 6) if has_geo else None
    lon = round(lon_c + np.random.normal(0, spread * 0.4), 6) if has_geo else None

    descriptors = DESCRIPTORS.get(complaint_type, DESCRIPTORS["DEFAULT"])

    return {
        "complaint_id":           random_complaint_id(),
        "created_date":           created,
        "closed_date":            closed_date,
        "due_date":               created + timedelta(days=random.choice([3, 7, 14, 30])),
        "complaint_type":         complaint_type,
        "descriptor":             random.choice(descriptors),
        "status":                 status,
        "resolution_description": random.choice(RESOLUTION_TEMPLATES) if is_closed else None,
        "agency":                 COMPLAINT_AGENCY.get(complaint_type, "HPD"),
        "agency_name":            f"Dept of {COMPLAINT_AGENCY.get(complaint_type, 'Housing')}",
        "borough_clean":          borough,
        "city":                   borough.title(),
        "zip_code":               str(random.randint(10001, 11697)),
        "street_name":            f"{random.randint(1,200)} {random.choice(['Main St','Broadway','Park Ave','Elm St','Oak Ave'])}",
        "latitude":               lat,
        "longitude":              lon,
        "has_geo":                has_geo,
        "is_closed":              is_closed,
        "days_open":              days_open,
        "open_year":              created.year,
        "open_month":             created.month,
        "open_dow":               created.weekday() + 1,
        "_silver_at":             datetime.now(),
        "_is_synthetic":          True,
    }

## 3. Generate Records

In [0]:
print(f"Generating {NUM_RECORDS:,} synthetic complaints...")
records = [generate_record() for _ in range(NUM_RECORDS)]
synthetic_pdf = pd.DataFrame(records)

print(f"✓ Generated {len(synthetic_pdf):,} records")
print(f"  Status distribution:\n{synthetic_pdf['status'].value_counts()}")
print(f"  Borough distribution:\n{synthetic_pdf['borough_clean'].value_counts()}")

## 4. Write to Delta

In [0]:

synthetic_sdf = spark.createDataFrame(synthetic_pdf)

(
    synthetic_sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("open_year", "borough_clean")
    .saveAsTable(SYNTHETIC_TABLE)
)

print(f"✓ Synthetic table written: {SYNTHETIC_TABLE}")
print(f"  Row count: {spark.table(SYNTHETIC_TABLE).count():,}")



## 5. Optionally Merge Into Silver (for unified dev testing)

In [0]:
## MAGIC Uncomment to merge synthetic records into the main silver table.
MERGE_INTO_SILVER = False   # set True to combine

if MERGE_INTO_SILVER:
    from delta.tables import DeltaTable

    silver_dt = DeltaTable.forName(spark, SILVER_TABLE)
    syn_sdf   = spark.table(SYNTHETIC_TABLE).drop("_is_synthetic")

    silver_dt.alias("s").merge(
         syn_sdf.alias("n"),
         "s.complaint_id = n.complaint_id"
    ).whenNotMatchedInsertAll().execute()

    print(f"✓ Merged synthetic into {SILVER_TABLE}")

## 6. Quick Validation

In [0]:
from pyspark.sql import functions as F

display(
    spark.table(SYNTHETIC_TABLE)
    .groupBy("complaint_type")
    .agg(
        F.count("*").alias("count"),
        F.round(F.avg("days_open"), 1).alias("avg_days_open"),
        F.round(F.sum(F.col("is_closed").cast("int")) / F.count("*") * 100, 1).alias("close_rate_pct")
    )
    .orderBy(F.desc("count"))
)
